# MLP 2026 Term 1 OPPE 1
**Date:** 4th April 2026

In [ ]:
# @title Student's Details to be filled before the exam starts: {"run":"auto","vertical-output":true,"form-width":"100%","display-mode":"form"}

Student_Name = "PONNAM SRI SAI" # @param {type:"string"}
Student_Roll_No = "24f3003215" # @param {type:"string"}

try:
  if len(Student_Name) == 0:
    raise ValueError("Warning:  Please write your name")
  elif len(Student_Roll_No)==0:
    raise ValueError("Warning: Please write your roll number")

  else:
    print(f"Best Of Luck {Student_Name}({Student_Roll_No})")

except ValueError as e:
  print(e)

Best Of Luck PONNAM SRI SAI(24f3003215)


# Unit 1 of 2: Preprocessing

### Q. 1:

Which dataset are you using for this exam?

Data_preprocessing_V1

Data_preprocessing_V2

Data_preprocessing_V3

In [ ]:
import pandas as pd
import numpy as np

### Q. 2:


Find the number of outliers in the weight column based on the Inter Quartile range method to detect outliers.

For outliers only consider values strictly greater than or strictly less than the testing conditions only.


In [ ]:
df = pd.read_csv("/content/preprocessing_v1.csv")

In [ ]:
df.columns

Index(['engine_size', 'horsepower', 'mileage', 'age', 'weight', 'brand',
       'fuel_type', 'price'],
      dtype='object')

In [ ]:
iqr = df["weight"].quantile(0.75) - df["weight"].quantile(0.25)

lower_bound = df["weight"].quantile(0.25) - (1.5 * iqr)
upper_bound = df["weight"].quantile(0.75) + (1.5 * iqr)

outliers = df[(df["weight"] < lower_bound) | (df["weight"] > upper_bound)]
outliers.shape[0]

45

### Q. 3:

Find the ratio of missing values of horsepower to the non missing values in brand column. (Round upto 2 decimals)


In [ ]:
missing_values = df["horsepower"].isna().sum()

In [ ]:
ratio = missing_values / (df.shape[0] - missing_values)

In [ ]:
ratio

np.float64(0.44346431435445066)

### Q. 4:

Which of the following pairs of numerical features has the least absolute value of correlation coefficient?

mileage and age

age and weight

price and weight

mileage and price

In [ ]:
abs(df.corr(numeric_only=True))

,engine_size,horsepower,mileage,age,weight,price
engine_size,1.000000,0.002527,0.000079,0.016609,0.006255,0.320209
horsepower,0.002527,1.000000,0.018492,0.000486,0.021429,0.774867
mileage,0.000079,0.018492,1.000000,0.004240,0.018668,0.154185
age,0.016609,0.000486,0.004240,1.000000,0.005532,0.303576
weight,0.006255,0.021429,0.018668,0.005532,1.000000,0.107637
price,0.320209,0.774867,0.154185,0.303576,0.107637,1.000000


### Q. 5:

What is the difference between average weight of Petrol cars and Diesel cars? Take the absolute difference. (Round upto 2 decimals)


In [ ]:
df["fuel_type"].unique()

array(['Petrol', 'Diesel', 'Hybrid'], dtype=object)

In [ ]:
pet_avg_we = df[df["fuel_type"] == "Petrol"]["weight"].mean()
diesel_avg_we = df[df["fuel_type"] == "Diesel"]["weight"].mean()
abs(pet_avg_we - diesel_avg_we)

np.float64(2.635936319227312)

### Q. 6:




Perform the following filtering operations on the data and count the number of rows satisfying the following conditions:

1. fuel_type is either "Diesel" OR "Hybrid"
2. horsepower is not null
3. price is strictly greater than the median price of cars having the same fuel_type
4. mileage is strictly less than the brand-wise 60th percentile mileage
5. age is less than the overall median age

In [ ]:
mask = (
    (df["fuel_type"].isin(["Diesel", "Hybrid"]))
    & (df["horsepower"].notna())
    & (df["price"] > df.groupby("fuel_type")["price"].transform("median"))
    & (df["mileage"] < df.groupby("brand")["mileage"].transform(lambda x: x.quantile(0.6)))
    & (df["age"] < df["age"].median())
)

In [ ]:
df[mask].shape[0]

542

### Q. 7:

Perform data imputation in the following manner:

`Numerical values with mean value of the feature`

`Categorical values with the most frequent value`

After imputation is done which of the following columns has the highest median value?

engine_size

weight

age

mileage

horsepower

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

In [ ]:
X = df.copy()

In [ ]:
X.sample(5)

,engine_size,horsepower,mileage,age,weight,brand,fuel_type,price
1913,1.257972,171.901061,30796.994759,4.777104,1405.405338,Toyota,Petrol,29438.805841
7105,NaN,NaN,81755.834229,4.491854,958.979865,Honda,Petrol,30489.672706
3354,1.603740,181.277701,63896.776380,2.905657,949.464497,Honda,Petrol,32903.115964
5410,NaN,175.276014,55058.270066,5.649296,2190.236622,Toyota,Hybrid,32520.989059
6463,1.822013,NaN,69002.995201,3.677433,1721.587366,NaN,Petrol,28991.753422


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

numeric_imputer = SimpleImputer(strategy="mean")
categorical_imputer = SimpleImputer(strategy="most_frequent")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_imputer, numeric_cols),
        ("cat", categorical_imputer, categorical_cols)
    ]
)

X_preprocessed = preprocessor.fit_transform(X)


X_preprocessed.shape

(9000, 8)

In [ ]:
median_df = pd.DataFrame(X_preprocessed, columns=numeric_cols.tolist() + categorical_cols.tolist())
median_df

,engine_size,horsepower,mileage,age,weight,price,brand,fuel_type
0,1.013955,142.315813,56493.963671,-0.168256,1784.003366,33061.115389,BMW,Petrol
1,2.004316,161.325233,65336.523422,7.958609,1872.16737,24487.913249,Toyota,Petrol
2,2.004316,151.019894,47434.03016,2.019378,1471.316766,30199.579049,Toyota,Diesel
3,2.004316,135.157439,72441.2981,7.548236,930.691392,21803.494684,Toyota,Hybrid
4,2.004316,151.019894,63742.949738,3.690674,1351.592193,25798.895404,Toyota,Petrol
...,...,...,...,...,...,...,...,...
8995,1.712959,151.906916,81279.116901,3.795894,1539.079171,28141.103315,Honda,Petrol
8996,2.004316,157.482758,41014.409866,6.020913,1062.170643,28091.728507,Toyota,Petrol
8997,1.650293,105.701544,91775.304465,5.843027,1826.90249,26291.372218,BMW,Petrol
8998,2.004316,174.21618,59313.733385,2.753161,1636.393374,36025.807125,Honda,Hybrid


In [ ]:
median_df[["engine_size", "weight", "age", "mileage", "horsepower"]].median().argmax()

np.int64(3)

### Q. 8:

After imputation Scale the data in the following manner:

- `MinMaxScaler on weight,age,mileage`
- `StandardScaler on horespower,engine_size

` What ratio of the variance of "age" to the standard_deviation of "horsepower"? ( Round up to 3 decimals)

Var(age) / Std (horsepower)

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scaler = ColumnTransformer(
    transformers=[
        ("minmax", MinMaxScaler(), ["weight", "age", "mileage"]),
        ("standard", StandardScaler(), ["horsepower", "engine_size"])
], remainder = "passthrough")

X_scaled = scaler.fit_transform(median_df)

ratio_df = pd.DataFrame(X_scaled, columns=median_df.columns)
ratio_df


,engine_size,horsepower,mileage,age,weight,price,brand,fuel_type
0,0.669321,0.229632,0.472011,-0.260939,-2.3468,33061.115389,BMW,Petrol
1,0.706844,0.72557,0.528703,0.308943,0.0,24487.913249,Toyota,Petrol
2,0.536239,0.363131,0.413925,-0.0,0.0,30199.579049,Toyota,Diesel
3,0.306144,0.700527,0.574254,-0.47554,0.0,21803.494684,Toyota,Hybrid
4,0.485283,0.465121,0.518486,-0.0,0.0,25798.895404,Toyota,Petrol
...,...,...,...,...,...,...,...,...
8995,0.565079,0.471542,0.630916,0.026592,-0.690412,28141.103315,Honda,Petrol
8996,0.362103,0.607323,0.372766,0.19375,0.0,28091.728507,Toyota,Petrol
8997,0.687579,0.596467,0.698211,-1.358597,-0.838908,26291.372218,BMW,Petrol
8998,0.606497,0.40791,0.490089,0.695401,0.0,36025.807125,Honda,Hybrid


In [ ]:
ratio = ratio_df["age"].var() / ratio_df["horsepower"].std()
ratio

np.float64(8.066647845339805)

np.float64(0.12949151234160178)


### Q. 9:


After Scaling and Imputing Perform the Encoding of the data in the following manner:

- brand will be one hot encoded; remove the original brand column from the dataset
- fuel_type will be ordinally encoded

What is the difference between number of columns in new and old dataframe?


In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

pipe = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(), ["brand"]),
        ("ordinal", OrdinalEncoder(), ["fuel_type"])
    ], remainder = "passthrough")

X_encoded = pipe.fit_transform(ratio_df)
X_encoded.shape

(9000, 11)

In [ ]:
ratio_df

,engine_size,horsepower,mileage,age,weight,price,brand,fuel_type
0,0.669321,0.229632,0.472011,-0.260939,-2.3468,33061.115389,BMW,Petrol
1,0.706844,0.72557,0.528703,0.308943,0.0,24487.913249,Toyota,Petrol
2,0.536239,0.363131,0.413925,-0.0,0.0,30199.579049,Toyota,Diesel
3,0.306144,0.700527,0.574254,-0.47554,0.0,21803.494684,Toyota,Hybrid
4,0.485283,0.465121,0.518486,-0.0,0.0,25798.895404,Toyota,Petrol
...,...,...,...,...,...,...,...,...
8995,0.565079,0.471542,0.630916,0.026592,-0.690412,28141.103315,Honda,Petrol
8996,0.362103,0.607323,0.372766,0.19375,0.0,28091.728507,Toyota,Petrol
8997,0.687579,0.596467,0.698211,-1.358597,-0.838908,26291.372218,BMW,Petrol
8998,0.606497,0.40791,0.490089,0.695401,0.0,36025.807125,Honda,Hybrid


In [ ]:
X_encoded

array([[1.0, 0.0, 0.0, ..., -0.2609393714983371, -2.3467998233520047,
        33061.1153890834],
       [0.0, 0.0, 0.0, ..., 0.30894340391909364, 0.0, 24487.913249137862],
       [0.0, 0.0, 0.0, ..., -8.520534846297954e-16, 0.0,
        30199.57904917274],
       ...,
       [1.0, 0.0, 0.0, ..., -1.3585973314178152, -0.8389080938793206,
        26291.3722184058],
       [0.0, 0.0, 1.0, ..., 0.6954006855299978, 0.0, 36025.807125428764],
       [1.0, 0.0, 0.0, ..., -8.520534846297954e-16, -2.045361308900323,
        26430.0264661758]], dtype=object)

In [ ]:
shape_diff = X_encoded.shape[1] - ratio_df.shape[1]
shape_diff

3


### Q. 10:


5 points
After applying all the preprocessings in the above questions what is the average value of the `weight` feature?

Select the closest matching option

0.55

0.59

0.50

0.52

In [ ]:
print(0.55)

0.55


# Unit 2 of 2: Model Building

# Please regularly save your Answers in the exam portal by clicking on "Submit" button

We will now use this dataset for a regression task, where the price column will be treated as the label.


Separate the features and the label.

Split the data into 80% training set and 20% test set, using random_state = 0.

### Q.1:


Train a Ridge Model with alpha = 0.02 on the training dataset.

- Compute the explained_variance_score on the test data and enter its value. (Round upto 2 decimals)

In [ ]:
X = pd.read_csv("/content/Model_Building_v1.csv")

In [ ]:
X.sample(5)

,engine_size,horsepower,mileage,age,weight,fuel_type,brand_BMW,brand_Ford,brand_Honda,brand_Toyota,price
8637,-1.045783e+00,-0.062760,0.663427,0.681786,0.501773,0.0,1,0,0,0,29095.736650
170,2.162910e-02,0.000000,0.503386,0.515586,0.363690,1.0,0,0,1,0,32701.205304
2862,-1.018031e+00,0.000000,0.530775,0.504123,0.420021,0.0,0,0,0,1,30994.968413
8577,-1.164363e-15,-0.957665,0.458496,0.555411,0.424679,1.0,0,0,1,0,32008.430355
2162,-1.164363e-15,0.000000,0.705363,0.286655,0.411106,1.0,0,0,1,0,39076.341622


In [ ]:
y = X["price"]
X = X.drop("price", axis=1)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import explained_variance_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

In [ ]:
rid = Ridge(alpha=0.02)
rid.fit(X_train, y_train)

Ridge(alpha=0.02)

In [ ]:
y_preds = rid.predict(X_test)

explained_variance_score(y_test, y_preds)

0.6882989394509571

### Q. 2:


Find the weight (coefficient) of the horsepower feature given by the Ridge model. (Round to 2 decimals)


In [ ]:
rid.feature_names_in_

array(['engine_size', 'horsepower', 'mileage', 'age', 'weight',
       'fuel_type', 'brand_BMW', 'brand_Ford', 'brand_Honda',
       'brand_Toyota'], dtype=object)

In [ ]:
rid.coef_

array([  1607.58015696,   3350.83113487,  -6815.68679704, -10163.6235528 ,
         4410.22586393,   1275.84618812,   3005.38911593,  -1558.43358411,
         -525.60821589,   -921.34731645])

### Q. 3:


Train a SGDRegressor on the training datatset with the following hyperparameters:

`loss="squared_error"`,

`penalty="l2"`,

`alpha=0.0005`,

`max_iter=2000`,'

`learning_rate="adaptive"`

`random_state=0

Compute the rmse on the test set and enter the value. (Round to 2 decimals)

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
from math import sqrt

sgd = SGDRegressor(
    loss = 'squared_error',
    penalty = 'l2',
    alpha = 0.0005,
    max_iter =2000,
    learning_rate = 'adaptive',
    random_state = 0
)

sgd.fit(X_train, y_train)

y_preds = sgd.predict(X_test)

sqrt(mean_squared_error(y_test, y_preds))

3044.179714907816

### Q. 4:


Perform the following steps:

1.StandardScaling on the dataset

2.Creating polynomial features with degree = 2 (no bias)

3.Fitting a standard Linear Regression Model

Note : For all preprocessings fit and transform on the train data and only transform on the test data

Calculate the R2 score on the test dataset upto 2 decimal places.

(Dont use the scaled data for any other question unless specified -- use the origial train and test data for the other questions by default)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

scaler = StandardScaler()
poly = PolynomialFeatures(degree=2, include_bias=False)
lr = LinearRegression()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

lr.fit(X_train_poly, y_train)

y_preds = lr.predict(X_test_poly)

r2_score(y_test, y_preds)

0.6873238543844618

### Q. 5:


Train a Decision Tree Regressor on the training dataset using the following parameters:

Maximum Depth = 6

Minimum Samples Per Split = 11

Random State = 0

What are the number of nodes in the tree?

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(max_depth=6, min_samples_split=11, random_state=0)

tree.fit(X_train, y_train)

tree.tree_.node_count

125

In [ ]:
tree

DecisionTreeRegressor(max_depth=6, min_samples_split=11, random_state=0)

### Q. 6:



Which of the following features got the least importance assigned by the DecisionTree?

engine_size

weight

age

mileage

horsepower

brand_Ford

In [ ]:
X.columns

Index(['engine_size', 'horsepower', 'mileage', 'age', 'weight', 'fuel_type',
       'brand_BMW', 'brand_Ford', 'brand_Honda', 'brand_Toyota'],
      dtype='object')

In [ ]:
pd.DataFrame({"f":tree.feature_names_in_, 'i':tree.feature_importances_}).sort_values(by='i')

,f,i
7,brand_Ford,0.000000
8,brand_Honda,0.000000
9,brand_Toyota,0.000000
4,weight,0.001740
2,mileage,0.006118
5,fuel_type,0.012745
3,age,0.098966
0,engine_size,0.108976
6,brand_BMW,0.120822
1,horsepower,0.650632


### Q. 7:

Train a GradientBoosting Regressor with base estimator as a Decision Tree with random state = 0 on the training dataset.

Compute the Mean Absolute Error on the train set. (Round upto 2 decimals)


In [ ]:
?GradientBoostingRegressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

gbr = GradientBoostingRegressor(
    init=DecisionTreeRegressor(random_state=0),
    random_state=0
)

gbr.fit(X_train, y_train)

y_preds = gbr.predict(X_test)

mean_absolute_error(y_test, y_preds)

3547.3045674017803

### Q. 8:


5 points
Perform GridSearchCV on the training data using GradientBoosting Regressor using the following parameter grid:

Number of Estimators: [50, 100,150]

Learning Rate: [0.0001, 0.001]

Use CV=5 and negative mean squared error as scoring metric.

Keep random_state = 0

What is the optimal combination of learning rate and number of estimators?

learning_rate = 0.0001, n_estimators = 50

learning_rate = 0.001, n_estimators = 50

learning_rate = 0.001, n_estimators = 100

learning_rate = 0.001, n_estimators = 150

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.0001, 0.001]
}

grid_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=0),
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5
)

# grid_search.fit(X_train, y_train)

# grid_search.best_params_

### Q. 9:


Train an MLP Regressor on the training dataset with the following specifications:

Hidden Layers: 3 layers with sizes [50, 32, 16]
Activation Function: tanh

Solver: lbfgs

Random State: 0

Compute the Mean Squared Error on the test set (Round upto 2 decimal place)

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

mlp = MLPRegressor(
    hidden_layer_sizes=[50, 32, 16],
    activation='tanh',
    solver='lbfgs',
    random_state=0
)

mlp.fit(X_train, y_train)

y_preds = mlp.predict(X_test)

mean_squared_error(y_test, y_preds)

29706128.72441551

### Q. 10:

Train a K-Nearest Neighbors (KNN) Regressor on the training dataset.

- Test the model with k values in the range [5,7,11,14,20] using cross-validation (cross_val_score).

- Use r2 score as a scoring metric.

- Use cv = 5

Which value of `k` gives the best performance?

5

7

11

14

20

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score

knn = KNeighborsRegressor()
k_values = [5, 7, 11, 14, 20]

scores = []
for k in k_values:
    knn.n_neighbors = k
    score = cross_val_score(knn, X_train, y_train, cv=5, scoring='r2')
    scores.append(score.mean())
    print(f"k={k}: {score.mean()}")

best_k = k_values[scores.index(max(scores))]
best_k

k=5: 0.5759430824799731
k=7: 0.590050903223886
k=11: 0.5947163934391801
k=14: 0.5932092641653283
k=20: 0.5923468870794499


11

### Q. 11: